# PBx Score Leaderboard

**PBx** = maximum tokens-per-step (TPS) that achieves ≥ x% average score across all ParallelBench tasks.  
Higher PBx means the model can decode more tokens in parallel while maintaining quality.

Scores are computed via **linear interpolation** between adjacent (TPS, score) points.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("../results")

# Display names
MODEL_DISPLAY_NAMES = {
    "GSAI-ML__LLaDA-1.5": "LLaDA 1.5",
    "GSAI-ML__LLaDA-8B-Instruct": "LLaDA 8B Instruct",
    "Dream-org__Dream-v0-Instruct-7B": "Dream 7B",
    "apple__DiffuCoder-7B-Instruct": "DiffuCoder 7B",
    "Gen-Verse__TraDo-4B-Instruct": "TraDo 4B",
    "Gen-Verse__TraDo-8B-Instruct": "TraDo 8B",
}

METHOD_DISPLAY_NAMES = {
    "confidence_topk": "Confidence Top-K",
    "confidence_threshold": "Confidence Threshold",
    "confidence_factor": "Confidence Factor",
    "entropy_topk": "Entropy Top-K",
    "topk_margin": "Top-K Margin",
    "random": "Random",
    "left_to_right": "Left-to-Right",
    "origin": "Origin",
    "klass": "KLASS",
}

## Data Loading

Scan `results/` directory, extract rows compatible with `compute_pb_scores()`, and group by (model, unmasking method).

In [ ]:
# Category-level groups to exclude (only keep individual tasks + overall avg)
CATEGORY_GROUPS = {
    "parallelbench_puzzles",
    "parallelbench_text_writing",
    "parallelbench_waiting_line",
}


def load_all_rows(results_dir: Path) -> list[dict]:
    """Scan results/ and build rows compatible with compute_pb_scores().

    Each row contains: model, task, score, nfe, max_tokens, tokens_per_step,
    unmasking, k, steps, alg_threshold, alg_factor.
    """
    rows = []

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model = model_dir.name

        for method_dir in sorted(model_dir.iterdir()):
            if not method_dir.is_dir():
                continue
            method = method_dir.name

            for param_dir in sorted(method_dir.iterdir()):
                if not param_dir.is_dir():
                    continue
                # param = param_dir.name

                # Select latest timestamp directory
                timestamp_dirs = sorted([d for d in param_dir.iterdir() if d.is_dir()])
                if not timestamp_dirs:
                    continue
                latest_dir = timestamp_dirs[-1]

                result_file = latest_dir / "results_parallelbench.json"
                if not result_file.exists():
                    continue

                with open(result_file) as f:
                    data = json.load(f)

                config = data.get("config", {})
                cli_gen_kwargs = config.get("gen_kwargs") or {}
                results = data.get("results", {})
                task_configs = data.get("configs", {})

                for task_name, task_metrics in results.items():
                    if task_name in CATEGORY_GROUPS:
                        continue

                    score = task_metrics.get("score,none")
                    nfe = task_metrics.get("nfe,none")
                    tps = task_metrics.get("tokens_per_step,none")

                    if score is None or nfe is None:
                        continue

                    # Merge generation kwargs (CLI overrides task config)
                    task_gen_kwargs = task_configs.get(task_name, {}).get(
                        "generation_kwargs", {}
                    )
                    gen_kwargs = {**task_gen_kwargs, **cli_gen_kwargs}

                    max_tokens = gen_kwargs.get("max_tokens", "")
                    unmasking = gen_kwargs.get("unmasking", method)
                    k = gen_kwargs.get("k", "")
                    steps = gen_kwargs.get("steps", "")

                    # Derive k from max_tokens / steps if not explicit
                    if not k and max_tokens and steps:
                        try:
                            k = int(max_tokens) / int(steps)
                        except (ValueError, TypeError):
                            pass

                    row = {
                        "model": model,
                        "task": task_name,
                        "score": float(score),
                        "nfe": float(nfe),
                        "max_tokens": max_tokens,
                        "tokens_per_step": tps if tps is not None else "",
                        "unmasking": unmasking,
                        "k": k,
                        "steps": steps,
                        "alg_threshold": gen_kwargs.get("alg_threshold", ""),
                        "alg_factor": gen_kwargs.get("alg_factor", ""),
                    }
                    rows.append(row)

    return rows


all_rows = load_all_rows(RESULTS_DIR)
print(f"Loaded {len(all_rows)} rows")
print(f"Models: {sorted(set(r['model'] for r in all_rows))}")
print(f"Methods: {sorted(set(r['unmasking'] for r in all_rows))}")

## PBx Leaderboard — Per Model × Method

Compute PBx scores for each (model, unmasking method) combination and display as a ranked table.

In [ ]:
from parallelbench.analysis.pb_score import DEFAULT_THRESHOLDS, compute_pb_scores


def build_leaderboard(
    rows: list[dict],
    thresholds: list[int] | None = None,
) -> pd.DataFrame:
    """Build a PBx leaderboard DataFrame grouped by (model, unmasking method).

    Returns a DataFrame with columns: Model, Method, PB90, PB80, PB70, PB60.
    """
    if thresholds is None:
        thresholds = DEFAULT_THRESHOLDS

    # Group rows by (model, unmasking)
    groups: dict[tuple[str, str], list[dict]] = {}
    for row in rows:
        key = (row["model"], row["unmasking"])
        groups.setdefault(key, []).append(row)

    records = []
    for (model, unmasking), group_rows in sorted(groups.items()):
        pb_scores = compute_pb_scores(group_rows, thresholds=thresholds)
        record = {
            "Model": MODEL_DISPLAY_NAMES.get(model, model),
            "Method": METHOD_DISPLAY_NAMES.get(unmasking, unmasking),
        }
        for name, tps in pb_scores.items():
            record[name] = tps
        records.append(record)

    return pd.DataFrame(records)


leaderboard = build_leaderboard(all_rows)

# Sort by PB80 descending (primary ranking metric), then PB90
leaderboard_sorted = leaderboard.sort_values(
    ["PB80", "PB90"], ascending=[False, False], na_position="last"
).reset_index(drop=True)
leaderboard_sorted.index = leaderboard_sorted.index + 1  # 1-indexed rank
leaderboard_sorted.index.name = "Rank"

# Format: round to 1 decimal, replace NaN with "-"
display_df = leaderboard_sorted.copy()
for col in [f"PB{t}" for t in DEFAULT_THRESHOLDS]:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{x:.1f}" if pd.notna(x) else "-"
        )

display(display_df)

## Best Method per Model

For each model, show only the best-performing unmasking method (by PB80).

In [ ]:
# Best method per model (by PB80)
best_per_model = (
    leaderboard.sort_values("PB80", ascending=False, na_position="last")
    .drop_duplicates(subset=["Model"], keep="first")
    .sort_values("PB80", ascending=False, na_position="last")
    .reset_index(drop=True)
)
best_per_model.index = best_per_model.index + 1
best_per_model.index.name = "Rank"

best_display = best_per_model.copy()
for col in [f"PB{t}" for t in DEFAULT_THRESHOLDS]:
    if col in best_display.columns:
        best_display[col] = best_display[col].apply(
            lambda x: f"{x:.1f}" if pd.notna(x) else "-"
        )

display(best_display)

## PBx Bar Chart

Visual comparison of PB80 scores across all (model, method) combinations.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

# Method colors
METHOD_COLORS = {
    "Confidence Top-K": "#2563eb",
    "Confidence Threshold": "#7c3aed",
    "Confidence Factor": "#db2777",
    "Entropy Top-K": "#059669",
    "Top-K Margin": "#d97706",
    "Random": "#6b7280",
    "Left-to-Right": "#dc2626",
}

# Prepare data: filter out rows where PB80 is NaN, sort ascending for horizontal bar
plot_df = leaderboard_sorted.dropna(subset=["PB80"]).copy()
# Use raw numeric values (not the formatted display_df)
plot_df = (
    leaderboard.dropna(subset=["PB80"])
    .sort_values("PB80", ascending=True)
    .reset_index(drop=True)
)
plot_df["label"] = plot_df["Model"] + " / " + plot_df["Method"]

fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.4)))

colors = [METHOD_COLORS.get(m, "#94a3b8") for m in plot_df["Method"]]
bars = ax.barh(
    plot_df["label"], plot_df["PB80"], color=colors, edgecolor="white", linewidth=0.5
)

# Value labels
for bar, val in zip(bars, plot_df["PB80"]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center",
        fontsize=9,
    )

ax.set_xlabel("PB80 (TPS achieving ≥ 80% avg score)")
ax.set_title("PBx Leaderboard — PB80")
ax.set_xlim(0, plot_df["PB80"].max() * 1.15)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Grouped Bar Chart — All PBx Thresholds per Model (Best Method)

In [ ]:
import numpy as np

threshold_cols = [f"PB{t}" for t in DEFAULT_THRESHOLDS]
threshold_colors = {
    "PB90": "#dc2626",
    "PB80": "#2563eb",
    "PB70": "#059669",
    "PB60": "#d97706",
}

plot_best = best_per_model.sort_values(
    "PB80", ascending=False, na_position="last"
).reset_index(drop=True)
models = plot_best["Model"] + "\n(" + plot_best["Method"] + ")"

x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(max(8, len(models) * 2), 5))

for i, col in enumerate(threshold_cols):
    values = plot_best[col].fillna(0)
    bars = ax.bar(
        x + i * width,
        values,
        width,
        label=col,
        color=threshold_colors.get(col, "#94a3b8"),
    )
    for bar, val, raw in zip(bars, values, plot_best[col]):
        if pd.notna(raw) and val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylabel("Tokens per Step (TPS)")
ax.set_title("PBx Scores — Best Method per Model")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()